# 02: Recover the time order

**Owner:** C

**Purpose:** Approximately recover the order of the shuffled time windows from price tick sizes, then check the order using volatility, which was not used to build it. The result is used for the stationarity diagnostics, the ACF, and time-ordered validation.

**Input files:**
- `data/raw/book_train.parquet/`
- `data/raw/train.csv` (validation only, read after the order is saved)

**Output files:**
- `data/processed/time_order.parquet` (`time_id`, `time_rank`)
- `data/processed/tick_sizes.parquet` (intermediate, used only in this notebook)
- `report/time_order_validation.json`

**Limits:** The direction (forward or backward) and the gaps between windows (for example, overnight) cannot be identified. `time_rank` is an ordering, not a clock.

In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl

sys.path.append("..")

from src.time_order import (
    build_log_price_matrix,
    build_tick_table,
    lag_autocorrelation,
    nearest_neighbor_chain,
    pairwise_distances,
    path_length,
    spectral_order,
    standardize_columns,
    two_opt,
)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
REPORT = Path("../report")
SEED = 5650
results = {"seed": SEED}

## Step 1: Estimate tick sizes, one stock at a time

In [ ]:
start = time.time()
ticks = build_tick_table(RAW / "book_train.parquet")
ticks.write_parquet(PROCESSED / "tick_sizes.parquet")

price_dtype = str(
    pl.scan_parquet(RAW / "book_train.parquet/stock_id=0/*.parquet", hive_partitioning=False)
    .collect_schema()["bid_price1"]
)
tick_stats = ticks["tick_size"].describe()
print(f"Tick table: {ticks.height:,} rows in {time.time() - start:.0f} seconds")
print(f"Book price dtype: {price_dtype}")
print(tick_stats)

prices_by_stock = ticks.group_by("stock_id").agg(
    (0.01 / pl.col("tick_size")).median().alias("median_price")
)
results["ticks"] = {
    "rows": ticks.height,
    "price_dtype": price_dtype,
    "tick_min": ticks["tick_size"].min(),
    "tick_median": ticks["tick_size"].median(),
    "tick_max": ticks["tick_size"].max(),
    "stock_median_price_min": prices_by_stock["median_price"].min(),
    "stock_median_price_median": prices_by_stock["median_price"].median(),
    "stock_median_price_max": prices_by_stock["median_price"].max(),
}
print(results["ticks"])

## Step 2: Build and standardize the log real price matrix

In [ ]:
time_ids = np.sort(ticks["time_id"].unique().to_numpy())
log_price, stock_ids, missing = build_log_price_matrix(ticks, time_ids)
points = standardize_columns(log_price)

assert np.isfinite(points).all(), "Standardized matrix has non-finite values"
results["matrix"] = {
    "n_time_ids": int(log_price.shape[0]),
    "n_stocks": int(log_price.shape[1]),
    "cells_filled": int(missing.sum()),
    "cells_total": int(missing.size),
    "stocks_with_filled_cells": int((missing.sum(axis=0) > 0).sum()),
}
print(results["matrix"])

## Step 3: Order the windows

The spectral embedding gives the initial order. Two refinements are then compared, and the one with the shorter path is kept:

- **Chain:** a nearest-neighbor chain that starts at the first window in the spectral order, followed by 2-opt.
- **Spectral:** 2-opt applied directly to the spectral order.

The choice uses only Euclidean distance between price vectors. Volatility is not loaded until after the order is saved.

In [ ]:
distances = pairwise_distances(points)

start = time.time()
spectral = spectral_order(points, random_state=SEED)
spectral_seconds = time.time() - start

start = time.time()
chain = nearest_neighbor_chain(distances, start=int(spectral[0]))
chain_refined, chain_passes, chain_converged = two_opt(distances, chain)
chain_seconds = time.time() - start

start = time.time()
spectral_refined, spectral_passes, spectral_converged = two_opt(distances, spectral)
spectral_refined_seconds = time.time() - start

candidates = {"chain_then_2opt": chain_refined, "spectral_then_2opt": spectral_refined}
lengths = {name: path_length(distances, order) for name, order in candidates.items()}
chosen = min(lengths, key=lengths.get)
final_order = candidates[chosen]


def rank_agreement(order_a, order_b):
    """Return the absolute Spearman correlation between two orders' ranks."""
    rank_a = np.argsort(order_a)
    rank_b = np.argsort(order_b)
    return abs(float(np.corrcoef(rank_a, rank_b)[0, 1]))


results["order"] = {
    "spectral_n_neighbors": max(int(points.shape[0] / 10), 1),
    "path_length_spectral": path_length(distances, spectral),
    "path_length_chain": path_length(distances, chain),
    "path_length_chain_then_2opt": lengths["chain_then_2opt"],
    "path_length_spectral_then_2opt": lengths["spectral_then_2opt"],
    "chain_2opt_passes": chain_passes,
    "chain_2opt_converged": bool(chain_converged),
    "spectral_2opt_passes": spectral_passes,
    "spectral_2opt_converged": bool(spectral_converged),
    "chosen": chosen,
    "rank_agreement_two_refinements": rank_agreement(chain_refined, spectral_refined),
    "rank_agreement_chosen_vs_spectral": rank_agreement(final_order, spectral),
    "seconds_spectral": spectral_seconds,
    "seconds_chain_2opt": chain_seconds,
    "seconds_spectral_2opt": spectral_refined_seconds,
}
print(json.dumps(results["order"], indent=2))

In [ ]:
time_order = pl.DataFrame(
    {"time_id": time_ids[final_order], "time_rank": np.arange(len(final_order))},
    schema={"time_id": pl.Int32, "time_rank": pl.Int32},
)
assert time_order["time_id"].n_unique() == 3830
assert time_order["time_rank"].to_list() == list(range(3830))
time_order.write_parquet(PROCESSED / "time_order.parquet")
print(time_order.head())

## Step 4: Validate the order

Volatility was not used to build the order. Strong autocorrelation only under the recovered order is evidence that the order is real.

**Note:** The direction (forward or backward) and the gaps between windows (for example, overnight) cannot be identified.

In [ ]:
rng = np.random.default_rng(SEED)
n_permutations = 100
permutations = [rng.permutation(len(time_ids)) for _ in range(n_permutations)]

recovered_distance = float(distances[final_order[:-1], final_order[1:]].mean())
random_distances = np.array([distances[p[:-1], p[1:]].mean() for p in permutations])

train = pl.read_csv(RAW / "train.csv")
assert set(train["time_id"].unique().to_list()) == set(time_ids.tolist()), "time_ids differ"
market = (
    train.with_columns(pl.col("target").log().alias("log_target"))
    .group_by("time_id")
    .agg(pl.col("log_target").mean().alias("m"))
)
m_by_time = dict(zip(market["time_id"].to_list(), market["m"].to_list()))
m_id_order = np.array([m_by_time[t] for t in time_ids.tolist()])
m_recovered = m_id_order[final_order]

recovered_acf = lag_autocorrelation(m_recovered)
random_acf = np.array([lag_autocorrelation(m_id_order[p]) for p in permutations])

results["validation"] = {
    "n_permutations": n_permutations,
    "mean_distance_recovered": recovered_distance,
    "mean_distance_random_mean": float(random_distances.mean()),
    "mean_distance_random_q025": float(np.quantile(random_distances, 0.025)),
    "mean_distance_random_q975": float(np.quantile(random_distances, 0.975)),
    "distance_ratio_recovered_to_random": recovered_distance / float(random_distances.mean()),
    "lag1_acf_recovered": recovered_acf,
    "lag1_acf_random_mean": float(random_acf.mean()),
    "lag1_acf_random_q025": float(np.quantile(random_acf, 0.025)),
    "lag1_acf_random_q975": float(np.quantile(random_acf, 0.975)),
    "lag1_acf_random_max": float(random_acf.max()),
    "note": (
        "The direction (forward or backward) and the gaps between windows "
        "(for example, overnight) cannot be identified."
    ),
}
print(json.dumps(results["validation"], indent=2))

## Step 5: Could volatility leak into the order through the tick estimate?

Volatility is not an input, but it could still leak in. A quiet window visits fewer price levels, so its smallest gap can be larger than one tick. That would make its estimated price too low. If this happened across many stocks at once, windows with similar volatility could look similar in price and end up next to each other, which would inflate the autocorrelation above.

`Float32` prices add a second possible route. Rounding matters most for high-priced stocks, whose ticks are tiny, and windows that visit more price levels have more chances to produce a too-small gap.

Three checks follow, and none of them changes the saved order:

1. Compare each estimated price with a rolling median of its neighbors in the recovered order, and correlate that noise with `m_t`. A large absolute residual can also reflect real price moves, which are bigger when volatility is high, so this check alone cannot separate the two.
2. Count cells that are off by at least a factor of 1.5, which is what a missed tick would look like.
3. Rebuild the order separately from the lower-priced and higher-priced halves of the stocks. Then compare the two orders with each other, and compute `m_t`'s lag-1 autocorrelation under each.

In [ ]:
from scipy.ndimage import median_filter

ordered_points = points[final_order]
smoothed = median_filter(ordered_points, size=(11, 1), mode="nearest")
residuals = ordered_points - smoothed
mean_residual = residuals.mean(axis=1)
mean_abs_residual = np.abs(residuals).mean(axis=1)

log_ordered = log_price[final_order]
log_residuals = log_ordered - median_filter(log_ordered, size=(11, 1), mode="nearest")
missed_tick_cells = np.abs(log_residuals) > np.log(1.5)
missed_per_window = missed_tick_cells.sum(axis=1)


def order_from_subset(columns):
    """Build an order from selected stocks, using spectral order then 2-opt."""
    subset = standardize_columns(log_price[:, columns])
    subset_distances = pairwise_distances(subset)
    start_order = spectral_order(subset, random_state=SEED)
    refined, _, converged = two_opt(subset_distances, start_order)
    assert converged, "2-opt did not converge on a stock subset"
    return refined


median_log_price = np.median(log_price, axis=0)
low_priced = np.argsort(median_log_price)[: log_price.shape[1] // 2]
high_priced = np.argsort(median_log_price)[log_price.shape[1] // 2 :]
order_low = order_from_subset(low_priced)
order_high = order_from_subset(high_priced)

results["leak_check"] = {
    "rolling_median_window": 11,
    "corr_mt_mean_residual": float(np.corrcoef(m_recovered, mean_residual)[0, 1]),
    "corr_mt_mean_abs_residual": float(np.corrcoef(m_recovered, mean_abs_residual)[0, 1]),
    "cells_off_by_factor_1_5": int(missed_tick_cells.sum()),
    "share_cells_off_by_factor_1_5": float(missed_tick_cells.mean()),
    "corr_mt_cells_off_per_window": float(np.corrcoef(m_recovered, missed_per_window)[0, 1]),
    "low_priced_max_median_price": float(np.exp(median_log_price[low_priced].max())),
    "high_priced_min_median_price": float(np.exp(median_log_price[high_priced].min())),
    "lag1_acf_order_from_low_priced": lag_autocorrelation(m_id_order[order_low]),
    "lag1_acf_order_from_high_priced": lag_autocorrelation(m_id_order[order_high]),
    "rank_agreement_low_vs_high": rank_agreement(order_low, order_high),
    "rank_agreement_final_vs_low": rank_agreement(final_order, order_low),
    "rank_agreement_final_vs_high": rank_agreement(final_order, order_high),
}
print(json.dumps(results["leak_check"], indent=2))

In [ ]:
(REPORT / "time_order_validation.json").write_text(json.dumps(results, indent=2) + "\n")
print("Saved", REPORT / "time_order_validation.json")